# Silver to Gold: Item Change Exploration
This notebook explores item-level silver parquet files and prepares API-friendly gold outputs.

Focus:
- Load all silver item parquet files
- Validate shape and quality
- Build API-ready views by patch and by item
- Export gold datasets for serving

In [1]:
from pathlib import Path
import json
import polars as pl

In [2]:
SILVER_ROOT = Path("../data/silver/items")
GOLD_ROOT = Path("../data/gold/items")
GOLD_ROOT.mkdir(parents=True, exist_ok=True)

parquet_files = sorted(SILVER_ROOT.rglob("*.parquet"))
print(f"Silver root: {SILVER_ROOT}")
print(f"Parquet files discovered: {len(parquet_files)}")
if parquet_files:
    print("Example:", parquet_files[0])

Silver root: ../data/silver/items
Parquet files discovered: 2212
Example: ../data/silver/items/patches/alpha/alpha_week_2/abyssal-scepter.parquet


In [3]:
rows = []
for file_path in parquet_files:
    rel = file_path.relative_to(SILVER_ROOT)
    parts = rel.parts
    item_slug = file_path.stem

    # Expected shape: patches/<era>/<patch_id>/<item>.parquet
    dataset = parts[0] if len(parts) > 0 else None
    era = parts[1] if len(parts) > 1 else None
    patch_id = parts[2] if len(parts) > 2 else None

    df = pl.read_parquet(file_path)
    row = df.to_dicts()[0]
    row["silver_rel_path"] = str(rel)
    row["dataset"] = dataset
    row["era"] = era
    row["patch_id"] = patch_id
    row["item_slug"] = item_slug
    rows.append(row)

silver_df = pl.DataFrame(rows) if rows else pl.DataFrame()
print(silver_df.shape)
silver_df.head(5)

(2212, 11)


item_name,matched_keywords,keyword_match_count,source_patch_dir,source_html,extracted_at_utc,silver_rel_path,dataset,era,patch_id,item_slug
str,list[str],i64,str,str,str,str,str,str,str,str
"""Abyssal Scepter""","[""Abyssal Scepter""]",1,"""patches/alpha/alpha_week_2""","""patches/alpha/alpha_week_2/02-…","""2026-04-02T09:33:52""","""patches/alpha/alpha_week_2/aby…","""patches""","""alpha""","""alpha_week_2""","""abyssal-scepter"""
"""Anathema's Chains""","[""heal""]",13,"""patches/alpha/alpha_week_2""","""patches/alpha/alpha_week_2/02-…","""2026-04-02T09:33:52""","""patches/alpha/alpha_week_2/ana…","""patches""","""alpha""","""alpha_week_2""","""anathema-s-chains"""
"""Bag of Tea""","[""Bag of Tea""]",1,"""patches/alpha/alpha_week_2""","""patches/alpha/alpha_week_2/02-…","""2026-04-02T09:33:52""","""patches/alpha/alpha_week_2/bag…","""patches""","""alpha""","""alpha_week_2""","""bag-of-tea"""
"""Bami's Cinder""","[""fire""]",1,"""patches/alpha/alpha_week_2""","""patches/alpha/alpha_week_2/02-…","""2026-04-02T09:33:52""","""patches/alpha/alpha_week_2/bam…","""patches""","""alpha""","""alpha_week_2""","""bami-s-cinder"""
"""Black Cleaver""","[""Black Cleaver""]",1,"""patches/alpha/alpha_week_2""","""patches/alpha/alpha_week_2/02-…","""2026-04-02T09:33:52""","""patches/alpha/alpha_week_2/bla…","""patches""","""alpha""","""alpha_week_2""","""black-cleaver"""


In [4]:
# Dataset quality and coverage checks
print("Total rows:", silver_df.height)
print("Unique patches:", silver_df.select(pl.col("patch_id").n_unique()).item())
print("Unique items:", silver_df.select(pl.col("item_name").n_unique()).item())

silver_df.group_by("era").agg(pl.len().alias("item_change_rows")).sort("era")

Total rows: 2212
Unique patches: 103
Unique items: 156


era,item_change_rows
str,u32
"""alpha""",134
"""beta""",546
"""season_one""",1065
"""season_two""",467


In [5]:
# View 1: patch-centric payload (good for GET /patches/{patch_id})
patch_items_df = (
    silver_df.group_by(["era", "patch_id"])
    .agg(
        pl.col("item_name").sort().alias("items_changed"),
        pl.len().alias("item_change_count"),
        pl.col("silver_rel_path").sort().alias("source_files"),
    )
    .sort(["era", "patch_id"])
 )
patch_items_df.head(10)

era,patch_id,items_changed,item_change_count,source_files
str,str,list[str],u32,list[str]
"""alpha""","""alpha_week_2""","[""Abyssal Scepter"", ""Anathema's Chains"", … ""World Atlas""]",38,"[""patches/alpha/alpha_week_2/abyssal-scepter.parquet"", ""patches/alpha/alpha_week_2/anathema-s-chains.parquet"", … ""patches/alpha/alpha_week_2/world-atlas.parquet""]"
"""alpha""","""alpha_week_3""","[""Bami's Cinder"", ""Boots"", … ""Teleport""]",14,"[""patches/alpha/alpha_week_3/bami-s-cinder.parquet"", ""patches/alpha/alpha_week_3/boots-of-speed.parquet"", … ""patches/alpha/alpha_week_3/teleport.parquet""]"
"""alpha""","""alpha_week_4""","[""Anathema's Chains"", ""Avarice Blade"", … ""Terminus""]",22,"[""patches/alpha/alpha_week_4/anathema-s-chains.parquet"", ""patches/alpha/alpha_week_4/avarice-blade.parquet"", … ""patches/alpha/alpha_week_4/terminus.parquet""]"
"""alpha""","""alpha_week_5""","[""Abyssal Scepter"", ""Anathema's Chains"", … ""World Atlas""]",22,"[""patches/alpha/alpha_week_5/abyssal-scepter.parquet"", ""patches/alpha/alpha_week_5/anathema-s-chains.parquet"", … ""patches/alpha/alpha_week_5/world-atlas.parquet""]"
"""alpha""","""alpha_week_6""","[""Anathema's Chains"", ""Boots of Speed"", … ""Tiamat""]",20,"[""patches/alpha/alpha_week_6/anathema-s-chains.parquet"", ""patches/alpha/alpha_week_6/boots-of-speed.parquet"", … ""patches/alpha/alpha_week_6/tiamat.parquet""]"
"""alpha""","""alpha_week_7""","[""Anathema's Chains"", ""Archangel's Staff"", … ""World Atlas""]",18,"[""patches/alpha/alpha_week_7/anathema-s-chains.parquet"", ""patches/alpha/alpha_week_7/archangel-s-staff.parquet"", … ""patches/alpha/alpha_week_7/world-atlas.parquet""]"
"""beta""","""0.8.21.110""","[""Amplifying Tome"", ""Anathema's Chains"", … ""Tiamat""]",17,"[""patches/beta/0.8.21.110/amplifying-tome.parquet"", ""patches/beta/0.8.21.110/anathema-s-chains.parquet"", … ""patches/beta/0.8.21.110/tiamat.parquet""]"
"""beta""","""0.8.22.115""","[""Anathema's Chains"", ""Archangel's Staff"", … ""Vision Ward""]",24,"[""patches/beta/0.8.22.115/anathema-s-chains.parquet"", ""patches/beta/0.8.22.115/archangel-s-staff.parquet"", … ""patches/beta/0.8.22.115/vision-ward.parquet""]"
"""beta""","""0.9.22.16""","[""Aegis of the Legion"", ""Anathema's Chains"", … ""Zephyr""]",36,"[""patches/beta/0.9.22.16/aegis-of-the-legion.parquet"", ""patches/beta/0.9.22.16/anathema-s-chains.parquet"", … ""patches/beta/0.9.22.16/zephyr.parquet""]"


In [6]:
# View 2: item-centric timeline (good for GET /items/{item_name}/changes)
item_timeline_df = (
    silver_df.select([
        "item_name",
        "item_slug",
        "era",
        "patch_id",
        "matched_keywords",
        "keyword_match_count",
        "source_html",
        "source_patch_dir",
    ])
    .sort(["item_name", "era", "patch_id"])
)
item_timeline_df.head(10)

item_name,item_slug,era,patch_id,matched_keywords,keyword_match_count,source_html,source_patch_dir
str,str,str,str,list[str],i64,str,str
"""Abyssal Scepter""","""abyssal-scepter""","""alpha""","""alpha_week_2""","[""Abyssal Scepter""]",1,"""patches/alpha/alpha_week_2/02-…","""patches/alpha/alpha_week_2"""
"""Abyssal Scepter""","""abyssal-scepter""","""alpha""","""alpha_week_5""","[""Abyssal Scepter""]",1,"""patches/alpha/alpha_week_5/Wee…","""patches/alpha/alpha_week_5"""
"""Abyssal Scepter""","""abyssal-scepter""","""beta""","""2009-05-09""","[""Abyssal Scepter""]",1,"""patches/beta/2009-05-09/Releas…","""patches/beta/2009-05-09"""
"""Abyssal Scepter""","""abyssal-scepter""","""beta""","""2009-05-15""","[""Abyssal Scepter""]",1,"""patches/beta/2009-05-15/Releas…","""patches/beta/2009-05-15"""
"""Abyssal Scepter""","""abyssal-scepter""","""beta""","""2009-06-19""","[""Abyssal Scepter""]",1,"""patches/beta/2009-06-19/Releas…","""patches/beta/2009-06-19"""
"""Abyssal Scepter""","""abyssal-scepter""","""beta""","""2009-06-26""","[""Abyssal Scepter""]",1,"""patches/beta/2009-06-26/Releas…","""patches/beta/2009-06-26"""
"""Abyssal Scepter""","""abyssal-scepter""","""beta""","""2009-07-10""","[""Abyssal Scepter""]",1,"""patches/beta/2009-07-10/Releas…","""patches/beta/2009-07-10"""
"""Abyssal Scepter""","""abyssal-scepter""","""season_two""","""1.0.0.146""","[""Abyssal Scepter""]",1,"""patches/season_two/1.0.0.146/L…","""patches/season_two/1.0.0.146"""
"""Aegis of the Legion""","""aegis-of-the-legion""","""beta""","""0.9.22.16""","[""Aegis of the Legion""]",1,"""patches/beta/0.9.22.16/Release…","""patches/beta/0.9.22.16"""


In [7]:
# Example API document for one patch
if patch_items_df.height > 0:
    sample_patch = patch_items_df.row(0, named=True)
    print(json.dumps(sample_patch, indent=2, ensure_ascii=False))
else:
    print("No patch data available.")

# Example API document for one item timeline
if item_timeline_df.height > 0:
    sample_item = item_timeline_df.select("item_name").row(0)[0]
    sample_item_payload = (
        item_timeline_df.filter(pl.col("item_name") == sample_item)
        .to_dicts()
    )
    print(sample_item)
    print(json.dumps(sample_item_payload[:5], indent=2, ensure_ascii=False))
else:
    print("No item data available.")

{
  "era": "alpha",
  "patch_id": "alpha_week_2",
  "items_changed": [
    "Abyssal Scepter",
    "Anathema's Chains",
    "Bag of Tea",
    "Bami's Cinder",
    "Black Cleaver",
    "Blighting Jewel",
    "Bloodthirster",
    "Boots of Speed",
    "Chalice of Blessing",
    "Chalice of Harmony",
    "Death's Dance",
    "Edge of Finality",
    "Elixir of Brilliance",
    "Emblem of Valor",
    "Essence Reaver",
    "Faerie Charm",
    "Force of Nature",
    "Giant Slayer",
    "Guardian Angel",
    "Guardian's Horn",
    "Health Potion",
    "Infinity Edge",
    "Last Whisper",
    "Liandry's Anguish",
    "Mana Potion",
    "Meki Pendant",
    "Night Harvester",
    "Phantom Dancer",
    "Rabadon's Deathcap",
    "Radiant Virtue",
    "Regrowth Pendant",
    "Rod of Ages",
    "Serylda's Grudge",
    "Spirit Visage",
    "Staff of Flowing Water",
    "Terminus",
    "Thornmail",
    "World Atlas"
  ],
  "item_change_count": 38,
  "source_files": [
    "patches/alpha/alpha_week_2/abys

In [8]:
# Gold exports for API layer
silver_df.write_parquet(GOLD_ROOT / "item_changes_flat.parquet")
patch_items_df.write_parquet(GOLD_ROOT / "patch_items_aggregate.parquet")
item_timeline_df.write_parquet(GOLD_ROOT / "item_timeline.parquet")

# Optional lightweight JSON examples for contract discussions
patch_items_df.head(100).write_ndjson(GOLD_ROOT / "patch_items_sample.ndjson")
item_timeline_df.head(200).write_ndjson(GOLD_ROOT / "item_timeline_sample.ndjson")

print("Gold outputs written to:", GOLD_ROOT)
print(sorted(p.name for p in GOLD_ROOT.iterdir()))

Gold outputs written to: ../data/gold/items
['item_changes_flat.parquet', 'item_timeline.parquet', 'item_timeline_sample.ndjson', 'patch_items_aggregate.parquet', 'patch_items_sample.ndjson']


In [9]:
# Endpoint-shaped JSON materialization for static API serving tests
API_ROOT = GOLD_ROOT / "api"
PATCH_API_DIR = API_ROOT / "patches"
ITEM_API_DIR = API_ROOT / "items"
PATCH_API_DIR.mkdir(parents=True, exist_ok=True)
ITEM_API_DIR.mkdir(parents=True, exist_ok=True)

# One JSON per patch_id
patch_docs_written = 0
for row in patch_items_df.to_dicts():
    patch_id = row["patch_id"]
    payload = {
        "patch_id": patch_id,
        "era": row["era"],
        "item_change_count": row["item_change_count"],
        "items_changed": row["items_changed"],
        "source_files": row["source_files"],
    }
    out_file = PATCH_API_DIR / f"{patch_id}.json"
    out_file.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    patch_docs_written += 1

# One JSON per item_slug with timeline entries
item_docs_written = 0
for item_slug in item_timeline_df.select("item_slug").unique().to_series().to_list():
    subset = (
        item_timeline_df
        .filter(pl.col("item_slug") == item_slug)
        .sort(["era", "patch_id"])
        .to_dicts()
    )
    if not subset:
        continue
    payload = {
        "item_slug": item_slug,
        "item_name": subset[0]["item_name"],
        "change_count": len(subset),
        "timeline": subset,
    }
    out_file = ITEM_API_DIR / f"{item_slug}.json"
    out_file.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    item_docs_written += 1

print(f"Patch endpoint docs: {patch_docs_written}")
print(f"Item endpoint docs: {item_docs_written}")
print("Patch sample:", sorted(p.name for p in PATCH_API_DIR.glob("*.json"))[:3])
print("Item sample:", sorted(p.name for p in ITEM_API_DIR.glob("*.json"))[:3])

Patch endpoint docs: 103
Item endpoint docs: 156
Patch sample: ['0.8.21.110.json', '0.8.22.115.json', '0.9.22.16.json']
Item sample: ['abyssal-scepter.json', 'aegis-of-the-legion.json', 'alacrity.json']


In [10]:
# Discovery indexes for quick endpoint lookup
patch_ids = patch_items_df.select("patch_id").unique().sort("patch_id").to_series().to_list()
item_slugs = item_timeline_df.select("item_slug").unique().sort("item_slug").to_series().to_list()

index_payload = {
    "patch_count": len(patch_ids),
    "item_count": len(item_slugs),
    "patch_ids": patch_ids,
    "item_slugs": item_slugs,
}

index_file = API_ROOT / "index.json"
index_file.write_text(json.dumps(index_payload, ensure_ascii=False, indent=2), encoding="utf-8")

# Optional split indexes
(API_ROOT / "patches_index.json").write_text(
    json.dumps({"patch_ids": patch_ids}, ensure_ascii=False, indent=2), encoding="utf-8"
)
(API_ROOT / "items_index.json").write_text(
    json.dumps({"item_slugs": item_slugs}, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Index files written:")
print(index_file)
print(API_ROOT / "patches_index.json")
print(API_ROOT / "items_index.json")
print(f"Patch IDs indexed: {len(patch_ids)}")
print(f"Item slugs indexed: {len(item_slugs)}")

Index files written:
../data/gold/items/api/index.json
../data/gold/items/api/patches_index.json
../data/gold/items/api/items_index.json
Patch IDs indexed: 103
Item slugs indexed: 156
